### Importation des librairies
Toutes les librairies sont centralisées dans un seul bloc afin de pouvoir les importer en une seule fois

In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from tqdm import tqdm
import warnings
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from joblib import Parallel, delayed

### Importation des fichiers

Importation de tous les fichiers nécessaires et affichage de leurs dimension au début pour les garder en mémoire pour l'exécution des différents algos par la suite. 

In [ ]:
x_train = pd.read_csv('data/x_train.csv', index_col=0)
x_test = pd.read_csv('data/x_test.csv', index_col=0)
y_train = pd.read_csv('data/y_train.csv', index_col=0)
sample_submission = pd.read_csv('data/new_output_sample.csv', index_col=0)

print("Dimensions x_train :", x_train.shape)
print("Dimensions y_train :", y_train.shape)
print("Dimensions x_test :", x_test.shape)
print("Dimensions sample_submission :", sample_submission.shape)

### Fonctions utiles
Ensemble de fonctions utiles dans la suite. Permet de rendre le code plus clair en évitant les répétitions

In [ ]:
def generer_soumission(soumission, nom_sortie):
    """
    Génère le fichier CSV à soumettre sur la page du challenge après l'avoir vérifié. 
    Si le fichier n'est pas valide rien n'est exporté.
    Si le fichier est valide le fichier csv est créé avec le nom entré en paramètre.
    """
    is_valid = (soumission.shape == sample_submission.shape) # Vérification du format avec le format cible
    if is_valid:
        soumission.to_csv(f'{nom_sortie}.csv') # Génération du fichier de soumission selon le nom entré en paramètre
        print(f"Fichier '{nom_sortie}.csv' généré avec succès !") 
    else:
        # Si le fichier ne correspond pas à ce qui est demandé on ne génère rien
        print("Les dimensions ne correspondent pas au fichier sample.")

In [ ]:
def echantilloner (nb_echantillon):
    """
    Fonction d'échantillonage selon une taille d'échantillon entrée en paramètre.
    Il est possible d'utiliser les données complète en entrant None
    """
    # Si none alors pas d'échantillonage
    holed_cols = [col for col in x_test.columns if 'holed' in col]
    complete_cols = [col for col in x_test.columns if 'holed' not in col]
    x_test_filled = x_test[holed_cols].copy()

    if nb_echantillon != None : # Renvoie les données d'un échantillon de taille donnée (pour réduire le temps de calcul)
        X_features = x_test[complete_cols].sample(n=nb_echantillon, axis=1, random_state=67)
    else : # Renvoie les données complètes
        X_features = x_test[complete_cols]  
    return holed_cols, x_test_filled, X_features

### Soumission de base (interpolation linéaire)

Fonction d'interpolation linéaire (remplissage des trous par une ligne droite, comme lors des précédents TP, pour éviter une erreur nan)
C'est ce que fait le benchmark donné par le site

In [ ]:
def interpolation_lineaire(column):
    return column.interpolate(method='linear', limit_direction='both') # Evite les erreurs nan comme lors des derniers TP

In [ ]:
holed_cols_test = [col for col in x_test.columns if 'holed' in col]
# Application de l'algo
y_pred_test = x_test[holed_cols_test].apply(interpolation_lineaire, axis=0)
submission = y_pred_test.loc[sample_submission.index, sample_submission.columns]

generer_soumission(submission, "interpolation_lineaire") # Exportation du fichier

### Régression linéaire
Nous avons décidé de commencer par un algorithme de régression linéaire, comme nous l'avons étudié en TP. Pour tester la solution nous avons commencé avec un échantillon à 6000, permettant d'avoir un ordre d'idée du score obtenu par l'algo sans perdre trop de temps. Au bout d'une vingtaine de minute de calcul nous avons obtenu un score de 98.

Une fois l'algorithme exécuté sur l'ensemble des données (code ci-dessous), le score obtenu a été de 93 pour un temps d'exécution d'environ 1h.

La régression linéaire semble donc bien fonctionner, notamment quand on lui donne un maximum de donnée. Néanmoins il est possible de faire bien mieux, comme on peut le voir sur le classement des scores.

In [ ]:
def regression_lineaire():

    holed_cols, x_test_filled, X_features_reduced = echantilloner(None) # Echantillonage
    print("Lancement de la régression linéaire sur l'échantillon...")

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        mask_known = target_series.notna() # Données connues
        mask_missing = target_series.isna() # Données à compléter
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
            
        # Enlève les messages d'erreur inutiles
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        top_vars = corrs.sort_values(ascending=False).head(50).index
        
        model = LinearRegression() # Modèle de régression linéaire
        model.fit(X_features_reduced.loc[mask_known, top_vars], target_series[mask_known])        
        predictions = model.predict(X_features_reduced.loc[mask_missing, top_vars])
        x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "regression_lineaire_full") # Génère le fichier CSV de rendu
    return submission

regression_lineaire()


### Arbres
Après la régression linéaire nous avons testé la méthode des arbres, également vue en TP. Avec un échantillon de 2000 (ayant pris plus de temps qu'un échantillon de 6000 en régression linéaire), nous avons obtenu un score de 106, ce qui est en dessous du benchmark.

Il s'agit de la méthode la moins efficace parmis les méthodes testées, bien qu'il soit certainement possible d'optimiser l'algo et obtenir de meilleurs score (en modifiant les paramètres ou l'algo en général) nous avons décidé d'abandonner cette solution (qui ne semble pas être la plus adaptée à ce contexte, comme nous le pensions).

In [ ]:
def regression_arbre_decision():
    holed_cols, x_test_filled, X_features_reduced = echantilloner(2000) # Echantillonage
    print("Lancement de l'Arbre de Décision (Sécurité anti-valeurs aberrantes)...")

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        mask_known = target_series.notna()
        mask_missing = target_series.isna()

        if mask_known.sum() < 2 or not mask_missing.any():
            continue    
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        top_vars = corrs.sort_values(ascending=False).head(15).index # On conserve les 15 meilleures valeurs
    
        # Application d'un algo avec une profondeur maximale à 6
        model = DecisionTreeRegressor(max_depth=6, random_state=67)    
        model.fit(X_features_reduced.loc[mask_known, top_vars], target_series[mask_known])
        predictions = model.predict(X_features_reduced.loc[mask_missing, top_vars])
        x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "arbre_decision_depth6") # Génération du fichier CSV à rendre
    return submission

soumission_arbre = regression_arbre_decision()

### Réseau de neurone
Pour terminer nous avons souhaité essayé les réseaux de neurones car nous pensions qu'ils seraient plus performants que la régression linéaire.

Nous avons commencé par un réseau de neuronnes utilisant la librairie Tensorflow, comme nous l'avions étudié en TP. L'algo nous a obtenu un score de 94 pour 1h30 de calcul (avec un échantillon à 6000). Ce score est moins intéressant que la régression linéaire.

Cependant, en faisant quelques recherches nous avons lu qu'utiliser Tensorflow ne serai ni la méthode la plus efficace, ni la méthode la plus performante pour un réseau de neuronnes dans ce contexte.

In [ ]:
def regression_reseau_neurones_opti():
    holed_cols, x_test_filled, X_features_reduced = echantilloner(6000)
    print("Lancement du Réseau de Neurones...")
    tf.get_logger().setLevel('ERROR')

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
        
        # N'affiche pas certains messages inutiles (affichage plus propre)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        top_vars = corrs.sort_values(ascending=False).head(50).index
        
        X_train_raw = X_features_reduced.loc[mask_known, top_vars].values
        y_train_nn = target_series[mask_known].values
        X_missing_raw = X_features_reduced.loc[mask_missing, top_vars].values
        
        scaler = StandardScaler() # Normalisation
        X_train_nn = scaler.fit_transform(X_train_raw)
        X_missing_nn = scaler.transform(X_missing_raw)
        
        model = Sequential([Dense(64, activation='relu', input_shape=(len(top_vars),)), Dense(32, activation='relu'), Dense(1)])
        model.compile(optimizer='adam', loss='mae') # On minimise la perte MAE (comme demandé)
        early_stop = EarlyStopping(monitor='loss', patience=10, verbose=0) # Arrêt du programme si il n'y a plus d'apprentissage
        model.fit(X_train_nn, y_train_nn, epochs=100, batch_size=32, verbose=0, callbacks=[early_stop])
        predictions = model.predict(X_missing_nn, verbose=0)
        x_test_filled.loc[mask_missing, target_col] = predictions.flatten()

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "reseau_neurones_keras_tensorflow")
    return submission

ma_soumission_finale = regression_reseau_neurones_opti()

Nous avons donc décidé d'utiliser Sklearn (une autre librairie utilisée en TP) et son algo de Gradient boost (méthode vue pour l'optimisation des réseaux de neuronnes en fin de cours), qui semble plus adaptée pour du texte et donc pour le contexte du challenge. 

Ce nouvel algo nous a permis d'obtenir un score de 83 en 25 min de calcul, soit 1h de moins qu'en utilisant Tensorflow pour 11 points gagnés, le tout sur un échantillon de 6000. De plus pour optimiser le résultat nous avons également ajouté dans le calcul leurs comme la moyenne, l'écart-type, le maximum et le minimum.

Comme pour la régression linéaire, nous avons décidé de faire tourner l'algorithme sur les données entières, mais malgré l'utilisation d'un ordinateur performant le temps de calcul était aux alentours de 1h30 à 2h environ alors que le processeur qui faisait les calculs n'était utilisé qu'à 30% maximum. 

In [ ]:

def regression_gradient():
    
    holed_cols, x_test_filled, X_features_reduced = echantilloner(None) # A modifier pour affinement
    print("Lancement du Gradient Boosting (Arbres) + Feature Engineering...")

    for target_col in tqdm(holed_cols):
        target_series = x_test[target_col]
        mask_known = target_series.notna()
        mask_missing = target_series.isna()
        
        if mask_known.sum() < 2 or not mask_missing.any():
            continue
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            corrs = X_features_reduced[mask_known].corrwith(target_series[mask_known]).abs()
        
        corrs = corrs.fillna(0)
        
        # 100 meilleures courbes (paramètre à modifier pour afiner)
        top_vars = corrs.sort_values(ascending=False).head(100).index
        
        # Extraction des données
        X_train_raw = X_features_reduced.loc[mask_known, top_vars].values
        y_train = target_series[mask_known].values
        X_missing_raw = X_features_reduced.loc[mask_missing, top_vars].values
        
        # Création de nouvelles colones pour optimiser l'algo
        def ajouter_features_statistiques(X):
            moyenne = np.mean(X, axis=1, keepdims=True)
            ecart_type = np.std(X, axis=1, keepdims=True)
            maximum = np.max(X, axis=1, keepdims=True)
            minimum = np.min(X, axis=1, keepdims=True)
            # Collage des nouvelles colonnes avec les autres courbes extraites précédemment
            return np.hstack((X, moyenne, ecart_type, maximum, minimum))
            
        X_train_enriched = ajouter_features_statistiques(X_train_raw)
        X_missing_enriched = ajouter_features_statistiques(X_missing_raw)
        
        # Application de l'algo de Gradient boost
        model = HistGradientBoostingRegressor(
            loss='absolute_error', # Minimise l'erreur
            max_iter=1000,
            learning_rate=0.03, # Taux d'apprentissage
            max_depth=6, # Arbres pas trop profonds
            
            early_stopping=True, # Limite le temps d'exécution en arrêtant le programme si l'apprentissage ne s'améliore plus
            validation_fraction=0.1, # Garde 10% des données pour tester l'arrêt
            n_iter_no_change=20,
            random_state=67
        )

        model.fit(X_train_enriched, y_train)
        predictions = model.predict(X_missing_enriched)
        x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "reseau_neurones_sklearn_full") # Génération du fichier CSV à soumettre
    
    return submission

soumission_gradient = regression_gradient()

### Verion parallèle du code
Afin d'avoir une exécution plus rapide du code, qui nous permet de tester plus de paramètres différents et faire plus de tests pour affiner le code, nous avons décidé de rendre l'exécution de l'algorithme parallèle afin d'utiliser le maximum des performances du processeur. 

Utiliser cette méthode nous a permis de passer de 1h30 de calcul environ à 30/40 min. Soit 3x plus rapide (Les temps de calculs dépendent bien évidemment des machines sur lesquelles nous travaillons).

Nous avons également profité de cette nouvelle itération pour améliorer et optimiser le code précédent en ajoutant des valeurs prises en compte dans l'algo (comme le percentile).

In [ ]:
def algo_gradient(target_col, x_test_global, X_features_global):
    # Fonction appliquant l'algo (très similaire à l'algo précédent)
    target_series = x_test_global[target_col]
    mask_known = target_series.notna()
    mask_missing = target_series.isna()
    
    if mask_known.sum() < 2 or not mask_missing.any():
        return target_col, None, None 
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        corrs = X_features_global[mask_known].corrwith(target_series[mask_known]).abs()
    
    corrs = corrs.fillna(0)
    top_vars = corrs.sort_values(ascending=False).head(100).index
    
    X_train_raw = X_features_global.loc[mask_known, top_vars].values
    y_train = target_series[mask_known].values
    X_missing_raw = X_features_global.loc[mask_missing, top_vars].values
    
    def ajouter_features_statistiques(X):
        # Inspiré de l'algo précédent mais ajout des percentiles et de la médiane pour optimiser les calculs
        moyenne = np.mean(X, axis=1, keepdims=True)
        mediane = np.median(X, axis=1, keepdims=True)
        ecart_type = np.std(X, axis=1, keepdims=True)
        maximum = np.max(X, axis=1, keepdims=True)
        minimum = np.min(X, axis=1, keepdims=True)
        q25 = np.percentile(X, 25, axis=1, keepdims=True)
        q75 = np.percentile(X, 75, axis=1, keepdims=True)
    
        return np.hstack((X, moyenne, mediane, ecart_type, maximum, minimum, q25, q75))
        
    X_train_enriched = ajouter_features_statistiques(X_train_raw)
    X_missing_enriched = ajouter_features_statistiques(X_missing_raw)
    
    # Application similaire à l'algo précédent mais avec quelques modifications des paramètres
    model = HistGradientBoostingRegressor(
        loss='absolute_error', 
        max_iter=3000, # Augmentation du nombre d'itération maximale (sans passer dans de l'overlifting)
        learning_rate=0.01, # Apprentissage encore plus lent 
        max_depth=6, 
        l2_regularization=0.5, # Ajout d'un nouveau paramètre pour éviter l'overlifting)

        early_stopping=True, # On conserve l'arrêt automatique
        validation_fraction=0.1,
        n_iter_no_change=40, # Plus de patience avant l'arrêt
        random_state=67
    )
    
    model.fit(X_train_enriched, y_train)
    predictions = model.predict(X_missing_enriched)
    
    return target_col, predictions, mask_missing

# Fonction principale qui applique l'algo de manière parallèle
def gradient_boosting_parallele():
    
    holed_cols, x_test_filled, X_features_reduced = echantilloner(None) # Echanillonage
    print("Lancement du Gradient Boosting en parallèle")

    # Traitement en parallèle (-1 pour utiliser le processeur complètement)
    resultats = Parallel(n_jobs=-1)(
        delayed(algo_gradient)(col, x_test, X_features_reduced) for col in tqdm(holed_cols)
    )
    
    # Assemblage des données
    for target_col, predictions, mask_missing in resultats:
        if predictions is not None:
            x_test_filled.loc[mask_missing, target_col] = predictions

    submission = x_test_filled.loc[sample_submission.index, sample_submission.columns]
    generer_soumission(submission, "Gradient_boosting_Parallele_100_Courbes") # Génerer le fichier CSV de soumission
    
    return submission

soumission_gradient_parallele = gradient_boosting_parallele()